In [2]:
import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import random_clifford, Pauli

# 1) Generate a random 2-qubit Clifford and obtain its circuit
cliff = random_clifford(2)
qc = cliff.to_circuit()
print("Random 2-qubit Clifford circuit:")
print(qc)

# 2) Simulate the unitary action via AerSimulator
sim = AerSimulator(method='unitary')

# Add save_unitary instruction to save the unitary matrix
qc.save_unitary()

tqc = transpile(qc, sim)
job = sim.run(tqc)
result = job.result()

# Extract the 4x4 unitary
U = np.asarray(result.get_unitary(qc))

# 3) Build the full two-qubit Pauli basis
pauli_labels = [
    'II','IX','IY','IZ',
    'XI','XX','XY','XZ',
    'YI','YX','YY','YZ',
    'ZI','ZX','ZY','ZZ'
]
pauli_mats = {lab: Pauli(lab).to_matrix() for lab in pauli_labels}

# 4) Conjugate the Z/I strings and identify outputs up to phase
inputs = ['II', 'IZ', 'ZI', 'ZZ']
print("\nMapping under U * P * U†:")
for lab in inputs:
    P = pauli_mats[lab]
    M = U @ P @ U.conj().T
    for cand, mat in pauli_mats.items():
        for phase in [1, -1, 1j, -1j]:
            if np.allclose(M, phase * mat):
                print(f"{lab} → {phase}·{cand}")
                break
        else:
            continue
        break

Random 2-qubit Clifford circuit:
     ┌───┐      ┌───┐ ┌───┐               
q_0: ┤ S ├──■───┤ H ├─┤ S ├──■────────────
     ├───┤┌─┴─┐┌┴───┴┐├───┤┌─┴─┐┌───┐┌───┐
q_1: ┤ X ├┤ X ├┤ Sdg ├┤ H ├┤ X ├┤ H ├┤ S ├
     └───┘└───┘└─────┘└───┘└───┘└───┘└───┘

Mapping under U * P * U†:
II → 1·II
IZ → 1·ZY
ZI → -1·IY
ZZ → -1·ZI


In [8]:
import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import random_clifford, Pauli
from itertools import product

def analyze_clifford_mapping(n_qubits, pauli_subset=None, print_circuit=False, verbose=False):
    """
    Analyze how a random n-qubit Clifford circuit transforms Pauli operators.
    
    Args:
        n_qubits: Number of qubits
        pauli_subset: List of Pauli strings to analyze. If None, uses all Z/I combinations.
    """
    
    # 1) Generate a random n-qubit Clifford and obtain its circuit
    cliff = random_clifford(n_qubits)
    qc = cliff.to_circuit()
    if print_circuit:
        print(f"Random {n_qubits}-qubit Clifford circuit:")
        print(qc)
    
    # 2) Simulate the unitary action via AerSimulator
    sim = AerSimulator(method='unitary')
    
    # Add save_unitary instruction to save the unitary matrix
    qc.save_unitary()
    
    tqc = transpile(qc, sim)
    job = sim.run(tqc)
    result = job.result()
    
    # Extract the 2^n × 2^n unitary
    U = np.asarray(result.get_unitary(qc))
    
    # 3) Build the full n-qubit Pauli basis
    pauli_ops = ['I', 'X', 'Y', 'Z']
    
    # Generate all possible n-qubit Pauli strings
    pauli_labels = [''.join(ops) for ops in product(pauli_ops, repeat=n_qubits)]
    pauli_mats = {lab: Pauli(lab).to_matrix() for lab in pauli_labels}
    
    # 4) Define input set - default to all Z/I combinations if not specified
    if pauli_subset is None:
        zi_ops = ['I', 'Z']
        inputs = [''.join(ops) for ops in product(zi_ops, repeat=n_qubits)]
    else:
        inputs = pauli_subset
    
    # 5) Conjugate the specified Pauli strings and identify outputs up to phase
    # print(f"\nMapping under U * P * U† for {len(inputs)} Pauli operators:")
    # print("-" * 50)
    result_mapping = {}
    for lab in inputs:
        if lab not in pauli_mats:
            print(f"Warning: {lab} is not a valid {n_qubits}-qubit Pauli string")
            continue
            
        P = pauli_mats[lab]
        M = U @ P @ U.conj().T
        
        found = False
        for cand, mat in pauli_mats.items():
            for phase in [1, -1, 1j, -1j]:
                if np.allclose(M, phase * mat, atol=1e-10):
                    phase_str = {1: "", -1: "-", 1j: "i", -1j: "-i"}[phase]
                    if verbose:
                        print(f"{lab} → {phase_str}{cand}")
                    result_mapping[lab] = (phase, cand)
                    found = True
                    break
            if found:
                break
        
        if not found:
            print(f"{lab} → (no match found - possible numerical error)")
    return result_mapping

def analyze_stabilizer_generators(n_qubits):
    """
    Analyze how a random Clifford transforms a typical stabilizer generator set.
    For n qubits, we use n independent Z operators as generators.
    """
    generators = []
    for i in range(n_qubits):
        # Create Pauli string with Z at position i, I elsewhere
        pauli_str = 'I' * i + 'Z' + 'I' * (n_qubits - i - 1)
        generators.append(pauli_str)
    
    print(f"\nAnalyzing stabilizer generators for {n_qubits} qubits:")
    print(f"Input generators: {generators}")
    
    analyze_clifford_mapping(n_qubits, pauli_subset=generators)

In [13]:
print("=" * 60)
analyze_clifford_mapping(2, verbose=True)

print("=" * 60)
analyze_clifford_mapping(3, verbose=True)

print("=" * 60)
analyze_clifford_mapping(3, verbose=True);

# # Example 2: 3-qubit case with default Z/I combinations
# print("\n" + "=" * 60)
# print("Example 2: 3-qubit case with all Z/I combinations")
# print("=" * 60)
# analyze_clifford_mapping(3)

# # Example 3: 4-qubit case
# print("\n" + "=" * 60)
# print("Example 3: 4-qubit stabilizer generators")
# print("=" * 60)
# analyze_clifford_mapping(4)

# # Example 4: Custom subset analysis
# print("\n" + "=" * 60)
# print("Example 4: 3-qubit custom Pauli subset")
# print("=" * 60)
# custom_paulis = ['III', 'ZZI', 'IZZ', 'XYZ', 'ZZZ']
# analyze_clifford_mapping(3, pauli_subset=custom_paulis)

II → II
IZ → -YI
ZI → YZ
ZZ → -IZ
III → III
IIZ → XZZ
IZI → -IZZ
IZZ → -XII
ZII → -XYX
ZIZ → -IXY
ZZI → XXY
ZZZ → IYX
III → III
IIZ → XIX
IZI → XZI
IZZ → IZX
ZII → ZYZ
ZIZ → -YYY
ZZI → -YXZ
ZZZ → -ZXY


In [ ]:
results4 = analyze_clifford_mapping(4)
{gen: results4[gen][1] for gen in ['ZIII', 'IZII', 'IIZI', 'IIIZ'] if gen in results4}

{'ZIII': 'YIIY', 'IZII': 'YYYI', 'IIZI': 'XYZX', 'IIIZ': 'YXXY'}

In [31]:
[s[1] for s in results4.values()]

['IIII',
 'YXXY',
 'XYZX',
 'ZZYZ',
 'YYYI',
 'IZZY',
 'ZIXX',
 'XXIZ',
 'YIIY',
 'IXXI',
 'ZYZZ',
 'XZYX',
 'IYYY',
 'YZZI',
 'XIXZ',
 'ZXIX']

In [26]:
results = analyze_clifford_mapping(5)
{gen: results[gen][1] for gen in ['ZIIII', 'IZIII', 'IIZII', 'IIIZI', 'IIIIZ'] if gen in results}

{'ZIIII': 'IIIZY',
 'IZIII': 'XXYII',
 'IIZII': 'XIYII',
 'IIIZI': 'YXZZY',
 'IIIIZ': 'IIIXX'}

In [2]:
import itertools
import networkx as nx
from typing import List, Tuple

# ---------- Symplectic Utilities ----------

def string_to_masks(pauli_string: str) -> Tuple[int, int]:
    """
    Converts a Pauli string into its binary symplectic representation.

    A Pauli operator P is represented by a pair of binary vectors (x, z).
    The representation is:
      I -> (x=0, z=0)
      X -> (x=1, z=0)
      Y -> (x=1, z=1)
      Z -> (x=0, z=1)

    For an n-qubit string, the vectors become n-bit integers. The i-th
    character corresponds to the (n-1-i)-th bit.

    Args:
        pauli_string: The string representation (e.g., "IXYZ").

    Returns:
        A tuple of two integers (x, z) representing the masks.
    """
    n = len(pauli_string)
    x_mask = 0
    z_mask = 0
    for i, char in enumerate(pauli_string):
        # The bit position corresponds to qubit index (n-1-i)
        bit_pos = n - 1 - i
        if char == 'X':
            x_mask |= (1 << bit_pos)
        elif char == 'Y':
            x_mask |= (1 << bit_pos)
            z_mask |= (1 << bit_pos)
        elif char == 'Z':
            z_mask |= (1 << bit_pos)
        # 'I' contributes nothing, so we skip it.
    return (x_mask, z_mask)

def do_anticommute(mask1: Tuple[int, int], mask2: Tuple[int, int]) -> bool:
    """
    Checks if two Pauli operators (in binary symplectic form) anticommute.

    Two operators P1 and P2 anticommute if their symplectic inner product is 1.
    The product is (x1.z2 + z1.x2) mod 2.

    Args:
        mask1: The (x, z) tuple for the first operator.
        mask2: The (x, z) tuple for the second operator.

    Returns:
        True if the operators anticommute, False otherwise.
    """
    x1, z1 = mask1
    x2, z2 = mask2
    
    # Calculate the dot products using bitwise AND
    # Calculate the sum using bitwise XOR
    product = (x1 & z2) ^ (z1 & x2)
    
    # The result is 1 (anticommute) if the popcount of the product is odd.
    # int.bit_count() is available in Python 3.10+
    # For older versions, use: bin(product).count('1')
    return product.bit_count() % 2 == 1

# ---------- Main Routine ----------

def largest_anticommuting_set(n: int, all_sets: bool = False) -> List:
    """
    Finds the largest set(s) of anticommuting n-qubit Pauli operators.

    Args:
        n: The number of qubits (must be > 0).
        all_sets: If True, return all largest sets. If False, return one.

    Returns:
        A list of Pauli strings for one largest set, or a list of such lists
        if all_sets is True.
    """
    if not isinstance(n, int) or n <= 0:
        raise ValueError("n must be a positive integer.")

    # 1. Enumerate all non-trivial Pauli words
    alphabet = ('I', 'X', 'Y', 'Z')
    identity_word = 'I' * n
    # Generate all 4^n possible strings
    all_words = ["".join(p) for p in itertools.product(alphabet, repeat=n)]
    # Filter out the identity operator
    words = [word for word in all_words if word != identity_word]

    # 2. Generate binary-symplectic masks for each word
    masks = [string_to_masks(w) for w in words]

    # 3. Build the anticommutation graph
    g = nx.Graph()
    # Add nodes, using indices to easily map back to words
    g.add_nodes_from(range(len(words)))
    
    # Add edges between any two nodes whose operators anticommute
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if do_anticommute(masks[i], masks[j]):
                g.add_edge(i, j)

    # 4. Ask networkx for a maximum clique
    # nx.find_cliques finds all maximal cliques. We need to find the largest ones.
    all_cliques = list(nx.find_cliques(g))
    
    if not all_cliques:
        return []

    # Find the size of the largest clique
    max_size = len(max(all_cliques, key=len))
    
    # Filter for cliques of that maximum size
    max_cliques_indices = [c for c in all_cliques if len(c) == max_size]
    
    # 5. Map indices back to Pauli strings and return
    result_sets = [[words[i] for i in clique] for clique in max_cliques_indices]
    
    if all_sets:
        return result_sets
    else:
        return result_sets[0]

# ---------- Quick Sanity Checks ----------
if __name__ == "__main__":
    print("--- Sanity Checks ---")

    # For 1 qubit, the largest set has size 3: {X, Y, Z}
    set_n1 = largest_anticommuting_set(1)
    print(f"\nLargest anticommuting set for n=1 (one solution):")
    # Sort for consistent output
    print(sorted(set_n1))
    # Expected: ['X', 'Y', 'Z']

    # For 2 qubits, the largest set has size 5
    all_sets_n2 = largest_anticommuting_set(2, all_sets=True)
    print(f"\nLargest anticommuting set for n=2:")
    print(f"Size of set: {len(all_sets_n2[0])}")
    print(f"Number of sets found: {len(all_sets_n2)}")
    print(f"Example set: {sorted(all_sets_n2[0])}")
    # Expected size: 5

    # For 3 qubits, the largest set also has size 5
    all_sets_n3 = largest_anticommuting_set(3, all_sets=True)
    print(f"\nLargest anticommuting set for n=2:")
    print(f"Size of set: {len(all_sets_n3[0])}")
    print(f"Number of sets found: {len(all_sets_n3)}")
    print(f"Example set: {sorted(all_sets_n3[0])}")

--- Sanity Checks ---

Largest anticommuting set for n=1 (one solution):
['X', 'Y', 'Z']

Largest anticommuting set for n=2:
Size of set: 5
Number of sets found: 6
Example set: ['IX', 'IY', 'XZ', 'YZ', 'ZZ']

Largest anticommuting set for n=2:
Size of set: 7
Number of sets found: 288
Example set: ['IIX', 'IIY', 'XIZ', 'YIZ', 'ZXZ', 'ZYZ', 'ZZZ']


In [ ]:
import itertools
import networkx as nx
from typing import List, Tuple, Sequence

# ---------- Symplectic Utilities (Unchanged) ----------

def string_to_masks(pauli_string: str) -> Tuple[int, int]:
    """
    Converts a Pauli string into its binary symplectic representation.
    """
    n = len(pauli_string)
    x_mask = 0
    z_mask = 0
    for i, char in enumerate(pauli_string):
        bit_pos = n - 1 - i
        if char == 'X':
            x_mask |= (1 << bit_pos)
        elif char == 'Y':
            x_mask |= (1 << bit_pos)
            z_mask |= (1 << bit_pos)
        elif char == 'Z':
            z_mask |= (1 << bit_pos)
    return (x_mask, z_mask)

def do_anticommute(mask1: Tuple[int, int], mask2: Tuple[int, int]) -> bool:
    """
    Checks if two Pauli operators (in binary symplectic form) anticommute.
    """
    x1, z1 = mask1
    x2, z2 = mask2
    product = (x1 & z2) ^ (z1 & x2)
    return product.bit_count() % 2 == 1



def largest_anticommuting_set(
    n: int, 
    all_sets: bool = False, 
    alphabet: Sequence[str] = ('I', 'X', 'Y', 'Z')
) -> List:
    """
    Finds the largest set(s) of anticommuting n-qubit Pauli operators
    drawn from a given alphabet.

    Args:
        n: The number of qubits (must be > 0).
        all_sets: If True, return all largest sets. If False, return one.
        alphabet: A tuple or list of characters to build operators from.
                  Defaults to the full Pauli group ('I', 'X', 'Y', 'Z').
                  To exclude Z, use ('I', 'X', 'Y').

    Returns:
        A list of Pauli strings for one largest set, or a list of such lists
        if all_sets is True.
    """
    if not isinstance(n, int) or n <= 0:
        raise ValueError("n must be a positive integer.")

    # 1. Enumerate all non-trivial words from the specified alphabet
    identity_word = 'I' * n
    all_words = ["".join(p) for p in itertools.product(alphabet, repeat=n)]
    words = [word for word in all_words if word != identity_word]

    # If there are no non-identity words, return an empty list
    if not words:
        return []

    # 2. Generate binary-symplectic masks for each word
    masks = [string_to_masks(w) for w in words]

    # 3. Build the anticommutation graph
    g = nx.Graph()
    g.add_nodes_from(range(len(words)))
    
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if do_anticommute(masks[i], masks[j]):
                g.add_edge(i, j)

    # 4. Ask networkx for the maximum clique(s)
    all_cliques = list(nx.find_cliques(g))
    if not all_cliques:
        # This can happen if n=1 and alphabet=('I', 'X'), for example.
        # The largest anticommuting set is just a single operator.
        return [words[0]] if not all_sets else [[w] for w in words]

    max_size = len(max(all_cliques, key=len))
    max_cliques_indices = [c for c in all_cliques if len(c) == max_size]
    
    # 5. Map indices back to Pauli strings and return
    result_sets = [[words[i] for i in clique] for clique in max_cliques_indices]
    
    if all_sets:
        return result_sets
    else:
        return result_sets[0]

print("--- Checks for the full Pauli Group {I, X, Y, Z} ---")

set_n1_full = largest_anticommuting_set(1)
print(f"\nLargest anticommuting set for n=1 (full group): {sorted(set_n1_full)}")
# Expected: ['X', 'Y', 'Z']

set_n2_full = largest_anticommuting_set(2)
print(f"Size of largest set for n=2 (full group): {len(set_n2_full)}")
# Expected: 5

print("\n" + "="*50 + "\n")
print("--- Checks for the Z-free Group {I, X, Y} ---")

# Define the Z-free alphabet
z_free_alphabet = ('I', 'X', 'Y')

# For 1 qubit, the largest set is {'X', 'Y'}
set_n1_zfree = largest_anticommuting_set(1, alphabet=z_free_alphabet)
print(f"\nLargest Z-free anticommuting set for n=1: {sorted(set_n1_zfree)}")
# Expected: ['X', 'Y']

# For 2 qubits, the largest set has size 3
all_sets_n2_zfree = largest_anticommuting_set(2, all_sets=True, alphabet=z_free_alphabet)
print(f"\nLargest Z-free anticommuting set for n=2:")
print(f"Size of set: {len(all_sets_n2_zfree[0])}")
print(f"Number of sets found: {len(all_sets_n2_zfree)}")
print(f"Example set: {sorted(all_sets_n2_zfree[0])}")
# Expected size: 3. Example: ['XX', 'XY', 'YI']

# For 3 qubits, the largest set also has size ...
set_n3_zfree = largest_anticommuting_set(3, alphabet=z_free_alphabet)
print(f"\nLargest Z-free anticommuting set for n=3:")
print(f"Size of set: {len(set_n3_zfree)}")
print(f"Example set: {sorted(set_n3_zfree)}")

# 4 qubits with Z-free alphabet
set_n4_zfree = largest_anticommuting_set(4, alphabet=z_free_alphabet)
print(f"\nLargest Z-free anticommuting set for n=4:")
print(f"Size of set: {len(set_n4_zfree)}")
print(f"Example set: {sorted(set_n4_zfree)}")

# 5 qubits with Z-free alphabet
set_n5_zfree = largest_anticommuting_set(5, alphabet=z_free_alphabet)
print(f"\nLargest Z-free anticommuting set for n=5:")
print(f"Size of set: {len(set_n5_zfree)}")
print(f"Example set: {sorted(set_n5_zfree)}")

# 6 qubits with Z-free alphabet
set_n6_zfree = largest_anticommuting_set(6, alphabet=z_free_alphabet)
print(f"\nLargest Z-free anticommuting set for n=6:")
print(f"Size of set: {len(set_n6_zfree)}")
print(f"Example set: {sorted(set_n6_zfree)}")

--- Checks for the full Pauli Group {I, X, Y, Z} ---

Largest anticommuting set for n=1 (full group): ['X', 'Y', 'Z']
Size of largest set for n=2 (full group): 5


--- Checks for the Z-free Group {I, X, Y} ---

Largest Z-free anticommuting set for n=1: ['X', 'Y']

Largest Z-free anticommuting set for n=2:
Size of set: 3
Number of sets found: 4
Example set: ['IX', 'XY', 'YY']

Largest Z-free anticommuting set for n=3:
Size of set: 5
Example set: ['IXX', 'XIY', 'XYX', 'YXY', 'YYI']

Largest Z-free anticommuting set for n=4:
Size of set: 8
Example set: ['IXXX', 'IYYY', 'XIXY', 'XXYI', 'XYIX', 'YIYX', 'YXIY', 'YYXI']

Largest Z-free anticommuting set for n=5:
Size of set: 9
Example set: ['IIIIX', 'IXXYY', 'IYYXY', 'XIXXY', 'XXYIY', 'XYIYY', 'YIYYY', 'YXIXY', 'YYXIY']


In [16]:
import itertools
import networkx as nx
from typing import List, Tuple, Sequence

# ---------- Symplectic Utilities ----------

def string_to_masks(pauli_string: str) -> Tuple[int, int]:
    """
    Converts a Pauli string into its binary symplectic representation.
    """
    n = len(pauli_string)
    x_mask = 0
    z_mask = 0
    for i, char in enumerate(pauli_string):
        bit_pos = n - 1 - i
        if char == 'X':
            x_mask |= (1 << bit_pos)
        elif char == 'Y':
            x_mask |= (1 << bit_pos)
            z_mask |= (1 << bit_pos)
        elif char == 'Z':
            z_mask |= (1 << bit_pos)
    return (x_mask, z_mask)

def do_anticommute(mask1: Tuple[int, int], mask2: Tuple[int, int]) -> bool:
    """
    Checks if two Pauli operators anticommute (symplectic product is 1).
    """
    x1, z1 = mask1
    x2, z2 = mask2
    product = (x1 & z2) ^ (z1 & x2)
    return product.bit_count() % 2 == 1

def do_commute(mask1: Tuple[int, int], mask2: Tuple[int, int]) -> bool:
    """
    Checks if two Pauli operators commute (symplectic product is 0).
    """
    # Two operators commute if they do not anticommute.
    return not do_anticommute(mask1, mask2)

# ---------- Main Routine ----------

def largest_centralizer_anticommuting_set(
    centralizer_string: str, 
    all_sets: bool = False, 
    alphabet: Sequence[str] = ('I', 'X', 'Y')
) -> List:
    """
    Finds the largest anticommuting set within the centralizer of a given
    Pauli string.

    Args:
        centralizer_string: The Pauli string to find the centralizer of.
        all_sets: If True, return all largest sets. If False, return one.
        alphabet: Characters to build operators from. Defaults to Z-free.

    Returns:
        A list of Pauli strings for one largest set, or a list of lists.
    """
    n = len(centralizer_string)
    if n == 0:
        raise ValueError("centralizer_string cannot be empty.")

    # 1. Generate all non-trivial words from the specified alphabet
    identity_word = 'I' * n
    all_words = ["".join(p) for p in itertools.product(alphabet, repeat=n)]
    candidate_words = [word for word in all_words if word != identity_word]

    # 2. Filter these words to find the centralizer set
    # The centralizer is the set of operators that commute with centralizer_string.
    center_mask = string_to_masks(centralizer_string)
    
    centralizer_set = []
    for word in candidate_words:
        word_mask = string_to_masks(word)
        if do_commute(center_mask, word_mask):
            centralizer_set.append(word)

    # The rest of the algorithm operates on this filtered 'centralizer_set'
    if not centralizer_set:
        return []

    # 3. Generate masks and build the anticommutation graph
    masks = [string_to_masks(w) for w in centralizer_set]
    g = nx.Graph()
    g.add_nodes_from(range(len(centralizer_set)))
    
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if do_anticommute(masks[i], masks[j]):
                g.add_edge(i, j)

    # 4. Find the maximum clique(s)
    all_cliques = list(nx.find_cliques(g))
    if not all_cliques:
        # No two operators anticommute, so the largest set is any single operator.
        return [centralizer_set[0]] if not all_sets else [[w] for w in centralizer_set]

    max_size = len(max(all_cliques, key=len))
    max_cliques_indices = [c for c in all_cliques if len(c) == max_size]
    
    # 5. Map indices back to Pauli strings and return
    result_sets = [[centralizer_set[i] for i in clique] for clique in max_cliques_indices]
    
    if all_sets:
        return result_sets
    else:
        return result_sets[0]

# ---------- Sanity Checks ----------
if __name__ == "__main__":
    z_free_alphabet = ('I', 'X', 'Y')

    print("--- Example 1: Centralizer of 'II' (Identity) ---")
    # The centralizer of Identity is the entire group.
    # This should be equivalent to the previous problem for n=2.
    set_n2_center_II = largest_centralizer_anticommuting_set(
        "II", all_sets=False, alphabet=z_free_alphabet
    )
    print(f"Centralizer: 'II', n=2")
    print(f"Size of largest anticommuting set: {len(set_n2_center_II)}")
    print(f"Example set: {sorted(set_n2_center_II)}")
    # Expected size: 3. Example: ['XX', 'XY', 'YI']
    print("-" * 40)

    print("--- Example 2: Centralizer of 'IX' ---")
    set_n2_center_IX = largest_centralizer_anticommuting_set(
        "IX", all_sets=False, alphabet=z_free_alphabet
    )
    print(f"Size of largest anticommuting set: {len(set_n2_center_IX)}")
    print(f"Example set: {sorted(set_n2_center_IX)}")
    # The centralizer of 'IX' is {'II', 'IX', 'YI', 'YX'}.
    # Within that, {YI, YX} is an anticommuting pair.
    # Expected size: 2. Example: ['YI', 'YX']
    print("-" * 40)
    
    print("--- Example 3: Centralizer of 'XX' ---")
    # This is a more complex case.
    all_sets_n2_center_XX = largest_centralizer_anticommuting_set(
        "XX", all_sets=True, alphabet=z_free_alphabet
    )
    print(f"Size of largest anticommuting set: {len(all_sets_n2_center_XX[0])}")
    print(f"Number of such sets: {len(all_sets_n2_center_XX)}")
    print(f"Example set: {sorted(all_sets_n2_center_XX[0])}")
    # Expected size: 3. Example: ['IY', 'XI', 'XX']
    print("-" * 40)
    print("--- Example 4: Centralizer of 'XZ' ---")
    all_sets_n2_center_XZ = largest_centralizer_anticommuting_set(
        "XZ", all_sets=True, alphabet=z_free_alphabet
    )
    print(f"Size of largest anticommuting set: {len(all_sets_n2_center_XZ[0])}")
    print(f"Number of such sets: {len(all_sets_n2_center_XZ)}")
    print(f"Example set: {sorted(all_sets_n2_center_XZ[0])}")
    print("-" * 40)

    # checks for n=3
    print("--- Example 5: Centralizer of 'III' (Identity) for n=3 ---")
    all_sets_n3_center_III = largest_centralizer_anticommuting_set(
        "III", all_sets=True, alphabet=z_free_alphabet)
    print(f"Size of largest anticommuting set: {len(all_sets_n3_center_III[0])}")
    print(f"Number of such sets: {len(all_sets_n3_center_III)}")
    print(f"Example set: {sorted(all_sets_n3_center_III[0])}")
    print("-" * 40)
    
    print("--- Example 6: Centralizer of 'IXX' for n=3 ---")
    all_sets_n3_center_IXX = largest_centralizer_anticommuting_set(
        "IXX", all_sets=True, alphabet=z_free_alphabet)
    print(f"Size of largest anticommuting set: {len(all_sets_n3_center_IXX[0])}")
    print(f"Number of such sets: {len(all_sets_n3_center_IXX)}")
    print(f"Example set: {sorted(all_sets_n3_center_IXX[0])}")
    print("-" * 40)

    print("--- Example 7: Centralizer of 'XYZ' for n=3 ---")
    all_sets_n3_center_XYZ = largest_centralizer_anticommuting_set(
        "XYZ", all_sets=True, alphabet=z_free_alphabet)
    print(f"Size of largest anticommuting set: {len(all_sets_n3_center_XYZ[0])}")
    print(f"Number of such sets: {len(all_sets_n3_center_XYZ)}")
    print(f"Example set: {sorted(all_sets_n3_center_XYZ[0])}")
    print("-" * 40)

--- Example 1: Centralizer of 'II' (Identity) ---
Centralizer: 'II', n=2
Size of largest anticommuting set: 3
Example set: ['IX', 'XY', 'YY']
----------------------------------------
--- Example 2: Centralizer of 'IX' ---
Size of largest anticommuting set: 2
Example set: ['XI', 'YI']
----------------------------------------
--- Example 3: Centralizer of 'XX' ---
Size of largest anticommuting set: 2
Number of such sets: 2
Example set: ['IX', 'YY']
----------------------------------------
--- Example 4: Centralizer of 'XZ' ---
Size of largest anticommuting set: 3
Number of such sets: 1
Example set: ['XI', 'YX', 'YY']
----------------------------------------
--- Example 5: Centralizer of 'III' (Identity) for n=3 ---
Size of largest anticommuting set: 5
Number of such sets: 8
Example set: ['IXX', 'XIY', 'XYX', 'YXY', 'YYI']
----------------------------------------
--- Example 6: Centralizer of 'IXX' for n=3 ---
Size of largest anticommuting set: 3
Number of such sets: 14
Example set: ['IYY

In [12]:
import numpy as np
# ------------------------------------------------------------
#  Pauli  ↔  binary symplectic vector  v = (a || b) ∈ F₂^{2n}
# ------------------------------------------------------------
_CHAR2BITS = {'I': (0, 0), 'X': (1, 0), 'Y': (1, 1), 'Z': (0, 1)}
_BITS2CHAR = {(0, 0): 'I', (1, 0): 'X', (1, 1): 'Y', (0, 1): 'Z'}

def pauli_to_vec(pstr: str) -> np.ndarray:
    pstr = pstr.replace(' ', '').upper()
    a = [_CHAR2BITS[c][0] for c in pstr]
    b = [_CHAR2BITS[c][1] for c in pstr]
    return np.array(a + b, dtype=np.uint8)

def vec_to_pauli(v: np.ndarray) -> str:
    n = v.size // 2
    a, b = v[:n], v[n:]
    return ''.join(_BITS2CHAR[int(x), int(z)] for x, z in zip(a, b))

# ------------------------------------------------------------
#  Symplectic algebra over F₂
# ------------------------------------------------------------
def J(n: int) -> np.ndarray:
    eye = np.eye(n, dtype=np.uint8)
    return np.block([[np.zeros_like(eye), eye],
                     [eye,               np.zeros_like(eye)]])

def symp_inner(v: np.ndarray, w: np.ndarray) -> int:
    n = v.size // 2
    a, b = v[:n], v[n:]
    a2, b2 = w[:n], w[n:]
    return int((a @ b2 + b @ a2) & 1)

def transvection_mat(u: np.ndarray) -> np.ndarray:
    s = (J(u.size // 2) @ u) & 1
    return (np.eye(u.size, dtype=np.uint8) ^ np.outer(u, s)) & 1

# ------------------------------------------------------------
#  Tiny GF(2) linear solver (row-reduced echelon form)
# ------------------------------------------------------------
def _gf2_solve(A: np.ndarray, b: np.ndarray):
    A, b = A.copy() & 1, b.copy() & 1
    m, n = A.shape
    piv, r = [], 0
    for c in range(n):
        idx = np.where(A[r:, c])[0]
        if idx.size == 0:
            continue
        i = idx[0] + r
        A[[r, i]], b[[r, i]] = A[[i, r]], b[[i, r]]
        piv.append(c)
        for j in range(m):
            if j != r and A[j, c]:
                A[j] ^= A[r]; b[j] ^= b[r]
        r += 1
        if r == m:
            break
    if np.any((A.sum(axis=1) == 0) & (b == 1)):
        return None          # no solution
    x = np.zeros(n, dtype=np.uint8)
    for row, c in reversed(list(enumerate(piv))):
        rhs = (A[row] @ x) & 1
        x[c] = b[row] ^ rhs
    return x

def _find_vector(constraints, n_qubits):
    JJ = J(n_qubits)
    A = np.vstack([(JJ @ v) & 1 for v, _ in constraints]).astype(np.uint8)
    b = np.array([val & 1 for _, val in constraints], dtype=np.uint8)
    return _gf2_solve(A, b)

# ------------------------------------------------------------
#  Map ONE vector, optionally keeping a set of vectors fixed
# ------------------------------------------------------------
def _map_one_vector(v, w, fixed, n_qubits):
    T_tot = np.eye(2 * n_qubits, dtype=np.uint8)
    gens   = []

    # (A) ensure v anti-commutes with w
    if symp_inner(v, w) == 0:
        constr = [(v, 1), (w, 1)] + [(f, 0) for f in fixed]
        t = _find_vector(constr, n_qubits)
        if t is None:
            raise ValueError("Could not find helper vector (should not happen).")
        T_t   = transvection_mat(t)
        T_tot = (T_t @ T_tot) & 1
        v     = v ^ t
        gens.append(t)

    # (B) single transvection now finishes the job
    u = v ^ w
    assert all(symp_inner(u, f) == 0 for f in fixed)
    T_u   = transvection_mat(u)
    T_tot = (T_u @ T_tot) & 1
    gens.append(u)
    return T_tot, gens

# ------------------------------------------------------------
#  Full 2-vector Clifford map
# ------------------------------------------------------------
def clifford_map(source_pair, target_pair):
    v1, v2 = [pauli_to_vec(p) for p in source_pair]
    w1, w2 = [pauli_to_vec(p) for p in target_pair]
    n      = len(source_pair[0].replace(' ', ''))

    if symp_inner(v1, v2) or symp_inner(w1, w2):
        raise ValueError("Both input pairs must commute.")

    # Step 1:  v1 → w1
    T1, gens1 = _map_one_vector(v1, w1, fixed=[], n_qubits=n)
    v2p       = (T1 @ v2) & 1            # updated second vector

    # Step 2:  v2p → w2   (keep w1 fixed)
    T2, gens2 = _map_one_vector(v2p, w2, fixed=[w1], n_qubits=n)

    S     = (T2 @ T1) & 1                # overall Clifford matrix
    imgs  = [vec_to_pauli((S @ v1) & 1), # *** original v1,v2 here ***
             vec_to_pauli((S @ v2) & 1)]

    return {
        "S"            : S,              # 2n×2n binary symplectic matrix
        "verify"       : imgs,           # should match target_pair
        "transvections": gens1 + gens2   # the u–vectors used
    }

# ------------------------------------------------------------
#  Quick self-test
# ------------------------------------------------------------
if __name__ == "__main__":
    src = ["IIXZ", "ZXII"]
    tgt = ["XXXX", "YYYZ"]
    out = clifford_map(src, tgt)
    print("Images :", out["verify"])     # →  ['ZII', 'IZI']


Images : ['XXXX', 'YYYZ']


# Just some quick tests

In [ ]:
import qutip
from src.quantum_utils import measure_povm, average_rho, average_rho_tensor_rho, POVMType, POVMElement
from src.quantum_utils import QuantumStatesBatch, QuantumKetsBatch, QuantumDensityMatricesBatch, QuantumOperator, POVM
from src.quantum_utils import sequence_of_qutip_qobj_to_states_batch, random_kets
from src.povms import sic_povm, mub_povm, random_rank1_povm
from src.qelm_utils import analyze_biasvar_vs_nstates, analyze_biasvar_vs_statistics
from src.QELM import QELM, TrainingData

from src.shadow_tomography import frame_operator
from src.utils import pp_matrix
from src import QELM


# train a QELM with a rank-1 POVM obtained from a random unitary
dim = 2
n_train_states = 100
# n_test_states = 50
train_stat = 100
num_outcomes = 20
povm = random_rank1_povm(dim=dim, num_outcomes=num_outcomes)
target_observable = QuantumOperator(qutip.sigmax().full())
train_states = QuantumKetsBatch(random_kets(dim=dim, num_kets=n_train_states))
test_states = QuantumKetsBatch(random_kets(dim=dim, num_kets=n_train_states))

train_frequencies = train_states.measure_povm(povm, statistics=train_stat, return_frequencies=True)
test_frequencies = test_states.measure_povm(povm, statistics=train_stat, return_frequencies=True)
train_expvals = train_states.expvals(observables=target_observable)
test_expvals = test_states.expvals(observables=target_observable)

train_data = TrainingData(frequencies=train_frequencies_, labels=train_expvals)
test_data = TrainingData(frequencies=test_frequencies_, labels=test_expvals)
qelm = QELM(train=train_data, test=test_data)
qelm = qelm.compute_MSE(train=True, test=True, display_results=False)

NameError: name 'TrainingData' is not defined

8